# Week 1: Tokenization Analysis

**Author:** Dalien Cable
**Course:** COSC 650 Applied LLM Systems

Measures the multilingual tax between English and European Portuguese using
a public-domain passage from Fernando Pessoa's *Livro do Desassossego*,
tokenized with OpenAI's tiktoken library (`cl100k_base` for GPT-4,
`o200k_base` for GPT-4o). No GPU or API key required.

See `README.md` in this folder for headline findings, and
`practice/tokenizer_exploration.py` for supporting investigation into why
GPT-4o compresses Portuguese better than GPT-4.

In [1]:
# Setup. In Colab, uncomment the install line on first run.
# !pip install tiktoken
import os, pathlib
os.environ['TIKTOKEN_CACHE_DIR'] = str((pathlib.Path('.') / '.tiktoken_cache').resolve())
os.makedirs(os.environ['TIKTOKEN_CACHE_DIR'], exist_ok=True)

import tiktoken
gpt4  = tiktoken.get_encoding('cl100k_base')   # GPT-4 / GPT-3.5
gpt4o = tiktoken.get_encoding('o200k_base')    # GPT-4o
print('tiktoken', tiktoken.__version__, '- encoders ready (cl100k_base, o200k_base)')

tiktoken 0.12.0 - encoders ready (cl100k_base, o200k_base)


## Helpers Functions

Two functions: count tokens for a string, and show the exact sub-token pieces a word breaks into.

In [2]:
def count_tokens(text, enc):
    return len(enc.encode(text))

def show_split(word, enc=gpt4):
    ids = enc.encode(word)
    pieces = [enc.decode([i]) for i in ids]
    print(f'{word!r:18s} -> {len(ids)} token(s): {pieces}')

# demo: some short strings are a single token; capitalized or rarer words fragment
for w in ['Panic', ' towel', '42', 'antidisestablishmentarianism']:
    show_split(w)

'Panic'            -> 2 token(s): ['P', 'anic']
' towel'           -> 1 token(s): [' towel']
'42'               -> 1 token(s): ['42']
'antidisestablishmentarianism' -> 6 token(s): ['ant', 'idis', 'establish', 'ment', 'arian', 'ism']


## Part 2: Selected passages

In [3]:
# Below is the opening paragraph of Portuguese Poet Fernando Pessoa's 'Gosto de dizer'.  This is available free at: https://pt.wikisource.org/wiki/G%C3%B3sto_de_dizer
# I verified the english translation 3 ways: Claude and Google Translations verified by my own Portugeuse literacy competency
english_text = "I like to speak. Or rather: I like to word things. To me, words are tangible bodies, visible sirens, embodied sensualities. Perhaps because actual sensuality holds no interest for me whatsoever — not even of the mind or of dreams — my desire has transmuted into a craving for that which creates verbal rhythms within me, or hears them in others. I shudder when things are well said. A certain page by Fialho, a certain page by Châteaubriand, makes my whole life tingle in every vein; it makes me seethe—tremulously still—with an unattainable pleasure that I am nonetheless experiencing. Even a page by Vieira, in its cold perfection of syntactic engineering, makes me tremble like a bough in the wind, in the passive delirium of something being moved."
foreign_text = "Gósto de dizer. Direi melhor: gósto de palavrar. As palavras são para mim corpos tocaveis, sereias visiveis, sensualidades incorporadas. Talvez porque a sensualidade real não tem para mim interesse de nenhuma especie — nem sequer mental ou de sonho —, transmudou-se-me o desejo para aquillo que em mim cria rhythmos verbaes, ou os escuta de outros. Estremeço se dizem bem. Tal pagina de Fialho, tal pagina de Châteaubriand, fazem formigar toda a minha vida em todas as veias, fazem-me raivar tremulamente quieto de um prazer inattingivel que estou tendo. Tal pagina, até, de Vieira, na sua fria perfeição de engenharia syntactica, me faz tremer como um ramo ao vento, num delirio passivo de coisa movida."

print('English words:', len(english_text.split()))
print('Foreign words:', len(foreign_text.split()))

def report(label, text):
    print(f'{label:9s} | chars {len(text):4d} | GPT-4 {count_tokens(text, gpt4):4d} | GPT-4o {count_tokens(text, gpt4o):4d}')

report('English', english_text)
report('Foreign', foreign_text)

tax_gpt4  = count_tokens(foreign_text, gpt4)  / count_tokens(english_text, gpt4)
tax_gpt4o = count_tokens(foreign_text, gpt4o) / count_tokens(english_text, gpt4o)
print(f'\nMultilingual tax  GPT-4: {tax_gpt4:.2f}x   GPT-4o: {tax_gpt4o:.2f}x')

# For my European Portuguese-English pairing, GPT-4 needs 21% more tokens for the same
#   text (210 vs. 174), while GPT-4o reduces that to 5% (180 vs. 172), even
#   though the Portuguese text has fewer words and characters than English.
# I tested why GPT-4o does better by running 14 words from this passage
#   through both tokenizers (see practice/tokenizer_exploration.py). Only
#   about a third of the diacritic words improved under GPT-4o, and
#   non-diacritic archaic spelled words improved at about the same rate.
#   Therefore diacritic handling isn't the main reason.
# More likely explanation: GPT-4o's vocabulary is roughly twice the size
#   (~200k vs ~100k tokens), giving it single-token coverage for more
#   Portuguese words. Which specific words get that coverage probably depends
#   on how often they appeared in its training data, not on any pattern I could evaluate.

English words: 128
Foreign words: 115
English   | chars  751 | GPT-4  174 | GPT-4o  172
Foreign   | chars  704 | GPT-4  210 | GPT-4o  180

Multilingual tax  GPT-4: 1.21x   GPT-4o: 1.05x


## Part 3: Evaluate with real figures

Turn the counts into engineering consequences. The skeleton below computes both; keep it pointed at your real passages.

In [4]:
CTX = 128_000
en = count_tokens(english_text, gpt4)
fo = count_tokens(foreign_text, gpt4)
print(f'A {CTX:,}-token window holds about {CTX//en:,} English copies and {CTX//fo:,} foreign copies of your passage.')
print(f'Per-request cost multiplier for the foreign language: {fo/en:.2f}x (billing is per token).')

# Given a 128k-token context window, this passage fits 735 times in English
#   but only 609 times in Portuguese. This is approx 17% less usable context for the same content.
# A product with a fixed context budget e.g. RAG retrieval system, runs out of room sooner for Portuguese.
#   This means fewer documents retrieved, less history kept, more gets cut off, reducing user experience
#   and potentially creating task based errors. In a medical setting this could cause harm to a patient.
# Billing per token, the same request costs 1.21x more in Portuguese than
#   English for identical information. This kind of language-based pricing
#   disparity functions like indirect discrimination, and it sits close to
#   two live regulatory trends: algorithmic pricing is a flagged 2026
#   enforcement priority, and laws like the EU AI Act and Colorado AI Act
#   already target algorithmic discrimination more broadly.

A 128,000-token window holds about 735 English copies and 609 foreign copies of your passage.
Per-request cost multiplier for the foreign language: 1.21x (billing is per token).


## Part 4: Bias splits and one failure

In [5]:
print('\nEuropean Portuguese bias pairs from passages in part 2 above:')
for port, eng in [('não', 'no'), ('palavrar', 'word'), ('engenharia', 'engineering')]:
    show_split(port)
    show_split(eng)

# These three pairs show the tokenizer's English-corpus bias directly:
#   'no' and 'word' are both single tokens (extremely common English words),
#   while their Portuguese equivalents fragment into 2-3 pieces each. The
#   'engenharia'/'engineering' pair is the clearest case: same Latin root,
#   same length, same meaning - English gets 1 token, Portuguese gets 3.
#   Since the words are structurally comparable, the gap doesn't look like it's about word complexity.
#   It could be related to how often each form showed up in training data
#   that potentially skews heavily to English.

print('\nArchaic vs modern spelling (same word, different century):')
for archaic, modern in [('syntactica', 'sintática'), ('pagina', 'página'), ('especie', 'espécie')]:
    print(f'\n{archaic!r} (pre-1911 Portuguese orthography):')
    show_split(archaic, gpt4)
    show_split(archaic, gpt4o)
    print(f'{modern!r} (modern spelling):')
    show_split(modern, gpt4)
    show_split(modern, gpt4o)
    
# Failure case: 
#   the modern spelling costs more tokens than the
#   outdated one. 'pagina' (pre-1911 Portuguese orthography, no accent) is 1 token in both
#   GPT-4 and GPT-4o. 'página' (modern, grammatically correct) is 2 tokens
#   in both, splitting at the accent. The modern word costs more than the outdated one.
#
# Why: 
#   Unclear, I tested whether 'pagina' is a token because of shared roots with 
#   English 'paginate'/'pagination' (see practice/tokenizer_exploration.py), but that was
#   disconfirmed: both those words tokenize as single whole-word tokens with no 'pagina' piece
#   involved, and 'paginated' splits as 'pag'+'inated', not touching
#   'pagina' either. Without access to OpenAI's training data, I can't
#   confirm why 'pagina' earned a merged token while 'página' didn't.
#
# Mitigation: 
#   Don't assume "correct," modern, or formal text is cheaper to
#   tokenize than casual or outdated text.  Test token counts on
#   real target text before budgeting or pricing. This example shows a system
#   shouldn't assume formal, grammatically correct input is automatically the
#   cheapest to process.  It has to measure the specific language patterns
#   it will see in production.


European Portuguese bias pairs from passages in part 2 above:
'não'              -> 2 token(s): ['n', 'ão']
'no'               -> 1 token(s): ['no']
'palavrar'         -> 3 token(s): ['pal', 'av', 'rar']
'word'             -> 1 token(s): ['word']
'engenharia'       -> 3 token(s): ['eng', 'enh', 'aria']
'engineering'      -> 1 token(s): ['engineering']

Archaic vs modern spelling (same word, different century):

'syntactica' (pre-1911 Portuguese orthography):
'syntactica'       -> 4 token(s): ['sy', 'nt', 'act', 'ica']
'syntactica'       -> 3 token(s): ['synt', 'act', 'ica']
'sintática' (modern spelling):
'sintática'        -> 3 token(s): ['s', 'int', 'ática']
'sintática'        -> 3 token(s): ['s', 'int', 'ática']

'pagina' (pre-1911 Portuguese orthography):
'pagina'           -> 1 token(s): ['pagina']
'pagina'           -> 1 token(s): ['pagina']
'página' (modern spelling):
'página'           -> 2 token(s): ['p', 'ágina']
'página'           -> 2 token(s): ['p', 'ágina']

'especie' (pr

## Conclusion

For this English-Portuguese pair, the multilingual tax was 1.21x under
GPT-4 and 1.05x under GPT-4o - most of that gap closes with the newer
tokenizer's larger vocabulary, not because Portuguese is inherently harder
to tokenize. In practice this means a 128k-token context window holds 735
English copies of the passage but only 609 Portuguese copies, and the same
request costs 1.21x more to run in Portuguese under GPT-4.

Three word pairs (não/no, palavrar/word, engenharia/engineering) showed
the tokenizer's English-corpus bias directly - engenharia/engineering is
the clearest case, since it's a cognate with the same root and length,
yet costs three tokens in Portuguese against one in English.

The failure case turned up something unexpected: the archaic, technically
outdated spelling "pagina" costs fewer tokens than the modern, correctly
accented "página" - token cost doesn't track linguistic correctness, and
I couldn't find its actual cause without access to the tokenizer's
training data (I tested and ruled out one plausible explanation - see
`practice/tokenizer_exploration.py`).

Overall, tokenization cost looks less like a fixed property of a language
and more like an artifact of what that tokenizer happened to see in
training - which is exactly why it needs to be measured directly rather
than assumed.